In [1]:
import numpy as np
import pandas as pd
import os
import scipy.io
from tensorflow import keras
from keras.utils import load_img,np_utils, img_to_array, to_categorical
from sklearn.model_selection import train_test_split
from keras.applications.vgg16 import VGG16
from keras.models import Model
from keras.layers import Dense, Flatten, Dropout
from keras.optimizers import Adam
from sklearn.model_selection import train_test_split
#from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [2]:
image_folder = 'flowers/'
label_file = 'imagelabels.mat'

In [3]:
labels = scipy.io.loadmat(label_file)
data = labels['labels']

In [4]:
num_classes = np.max(data) + 1
data = to_categorical(data, num_classes=num_classes)


In [5]:
data = data.reshape(data.shape[1:])
data.shape

(8189, 103)

In [6]:
image_files = os.listdir(image_folder)
print(len(image_files))
files = [item for item in image_files if ')' not in item]
print(len(files))
files1 = [item for item in files if 'ipynb_checkpoints' not in item]
print(len(files1))

8189
8189
8189


In [7]:
image_files = os.listdir(image_folder)
print(len(files1))
images = []
for filename in files1:
  img = load_img(image_folder+"/"+filename, target_size=(128, 128))  # VGG16 expects input of size 224x224
  img_array = img_to_array(img)
  images1 = np.expand_dims(img_array, axis=0)
  images.append(images1)
images = np.array(images)
print(images.shape)

8189
(8189, 1, 128, 128, 3)


In [8]:
images /= 255.
images = images.reshape(8189, 128, 128, 3)
images.shape

(8189, 128, 128, 3)

In [9]:
train_images, test_images, train_labels, test_labels = train_test_split(images, data,train_size=0.8 ,test_size=0.2, random_state=42)

In [10]:
print(train_images.shape)
print(train_labels.shape)
print(test_images.shape)
print(test_labels.shape)

(6551, 128, 128, 3)
(6551, 103)
(1638, 128, 128, 3)
(1638, 103)


In [11]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

for layer in base_model.layers:
    layer.trainable = False


x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
predictions = Dense(num_classes, activation='softmax')(x)

In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [13]:
model = Model(inputs=base_model.input, outputs=predictions)

In [14]:
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [15]:
model.fit(train_images, train_labels, epochs=20, validation_data=(test_images, test_labels))

Epoch 1/20
205/205 [==============================] - 263s 1s/step - loss: 4.4722 - accuracy: 0.0519 - val_loss: 4.1583 - val_accuracy: 0.1587
Epoch 2/20
205/205 [==============================] - 275s 1s/step - loss: 4.0105 - accuracy: 0.1319 - val_loss: 3.6578 - val_accuracy: 0.2436
Epoch 3/20
205/205 [==============================] - 284s 1s/step - loss: 3.5919 - accuracy: 0.2082 - val_loss: 3.2603 - val_accuracy: 0.3284
Epoch 4/20
205/205 [==============================] - 297s 1s/step - loss: 3.2320 - accuracy: 0.2763 - val_loss: 2.9515 - val_accuracy: 0.4090
Epoch 5/20
205/205 [==============================] - 318s 2s/step - loss: 2.9264 - accuracy: 0.3287 - val_loss: 2.6783 - val_accuracy: 0.4652
Epoch 6/20
205/205 [==============================] - 457s 2s/step - loss: 2.7067 - accuracy: 0.3751 - val_loss: 2.4899 - val_accuracy: 0.4969
Epoch 7/20
205/205 [==============================] - 485s 2s/step - loss: 2.5029 - accuracy: 0.4138 - val_loss: 2.3134 - val_accuracy: 0.5311

In [16]:
loss, accuracy = model.evaluate(test_images, test_labels)
print(f"Test Accuracy: {accuracy * 100}%")

52/52 [==============================] - 94s 2s/step - loss: 1.3484 - accuracy: 0.6947
Test Accuracy: 69.47497129440308%
